# 4월 데이터 검증 파이프라인 (Step 3 → 7)

**입력 파일**
- `data/4m_embedding_input.parquet` — 4월 전처리 결과
- `data/step4_2_umap_model.pkl` — 기존 UMAP 모델
- `data/step5_2_hdbscan_model.pkl` — 기존 HDBSCAN 모델
- `data/step6_1_topic_keywords.csv` — 기존 토픽 키워드
- `data/protected_words.json` — 보호어 사전

**출력 파일 (data2/4m_ prefix)**
- `4m_embeddings.npy`
- `4m_cluster_labels.npy`
- `4m_step7_1_sentences.parquet`
- `4m_step7_2_aspect_pairs.parquet`
- `4m_step7_3b_aspect_sentiment.parquet`
- `4m_step7_3b_review_sentiment.parquet`

## 0. 환경 설정

In [ ]:
!pip install -q sentence-transformers==2.7.0
!pip install -q umap-learn==0.5.6
!pip uninstall -y hdbscan
!pip install -q hdbscan
!pip install -q kiwipiepy
!pip install -q transformers accelerate

Found existing installation: hdbscan 0.8.42
Uninstalling hdbscan-0.8.42:
  Successfully uninstalled hdbscan-0.8.42
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 137.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

import os
import re
import json
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import hdbscan
from collections import Counter
from tqdm.auto import tqdm
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
입력: /content/drive/MyDrive/review_analysis/4m_data/4m_embedding_input.parquet


In [ ]:
drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
DATA2_DIR  = '/content/drive/MyDrive/sparta/tp4/review_analysis/month_data/'
INPUT_PATH = DATA2_DIR + '4m_embedding_input.parquet'

os.makedirs(DATA2_DIR, exist_ok=True)

print(f'DATA_DIR  : {DATA_DIR}')
print(f'DATA2_DIR : {DATA2_DIR}')
print(f'INPUT_PATH: {INPUT_PATH}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 1. Step 3 — 임베딩

In [ ]:
df = pd.read_parquet(INPUT_PATH)
df['리뷰내용_clean'] = df['리뷰내용_clean'].fillna('').astype(str)
df = df[df['리뷰내용_clean'].str.len() > 0].reset_index(drop=True)

docs = df['리뷰내용_clean'].tolist()
review_ids = df['리뷰번호'].tolist()

print(f'임베딩 대상: {len(df):,}건')

embedding_model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=device)

embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

np.save(DATA2_DIR + '4m_embeddings.npy', embeddings)
np.save(DATA2_DIR + '4m_review_ids.npy', np.array(review_ids))

print(f'임베딩 완료: {embeddings.shape}')
print(f'저장: 4m_embeddings.npy, 4m_review_ids.npy')

임베딩 대상: 2,752건


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

임베딩 완료: (2752, 768)
저장: 4m_embeddings.npy, 4m_review_ids.npy


In [ ]:
path = DATA_DIR + 'step4_2_umap_model.pkl'
print(f'파일 크기: {os.path.getsize(path):,} bytes')

파일 크기: 6,414,533,673 bytes


## 2. Step 4 — UMAP Transform (기존 모델 사용)

In [ ]:
# 기존 UMAP 모델 로드
with open(DATA_DIR + 'step4_2_umap_model.pkl', 'rb') as f:
    umap_model = pickle.load(f)

print('UMAP 모델 로드 완료')
print('Transform 중... (5~15분 소요)')

t0 = time.time()
umap_embeddings = umap_model.transform(embeddings)
elapsed = time.time() - t0

np.save(DATA2_DIR + '4m_umap_embeddings.npy', umap_embeddings)

print(f'UMAP transform 완료: {elapsed/60:.1f}분')
print(f'출력 shape: {umap_embeddings.shape}')
print(f'저장: 4m_umap_embeddings.npy')

Thu May  7 07:25:59 2026 Building and compiling search function
UMAP 모델 로드 완료
Transform 중... (5~15분 소요)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
UMAP transform 완료: 0.1분
출력 shape: (2752, 5)
저장: 4m_umap_embeddings.npy


## 3. Step 5 — 토픽 배정 (기존 모델 사용)

In [ ]:
# 기존 HDBSCAN 모델 로드
with open(DATA_DIR + 'step5_2_hdbscan_model.pkl', 'rb') as f:
    hdbscan_model = pickle.load(f)

print('HDBSCAN 모델 로드 완료')
print('토픽 배정 중...')

t0 = time.time()
cluster_labels, strengths = hdbscan.approximate_predict(hdbscan_model, umap_embeddings)
elapsed = time.time() - t0

np.save(DATA2_DIR + '4m_cluster_labels.npy', cluster_labels)
df['topic_id'] = cluster_labels

n_topics   = len(set(cluster_labels)) - 1
n_outliers = (cluster_labels == -1).sum()

print(f'토픽 배정 완료: {elapsed/60:.1f}분')
print(f'  토픽 수: {n_topics}개')
print(f'  아웃라이어: {n_outliers:,}건 ({n_outliers/len(cluster_labels)*100:.1f}%)')
print(f'  토픽 배정: {(cluster_labels != -1).sum():,}건')
print(f'저장: 4m_cluster_labels.npy')

HDBSCAN 모델 로드 완료
토픽 배정 중...
토픽 배정 완료: 0.0분
  토픽 수: 68개
  아웃라이어: 1,020건 (37.1%)
  토픽 배정: 1,732건
저장: 4m_cluster_labels.npy


## 4. Step 7-1 — 문장 분리

In [ ]:
# 보호어 로드
with open(DATA_DIR + 'protected_words.json', 'r', encoding='utf-8') as f:
    PROTECTED_WORDS = json.load(f)

all_protected = sum(PROTECTED_WORDS.values(), [])
print(f'보호어: {len(all_protected)}개')

kiwi = Kiwi()
for word in all_protected:
    kiwi.add_user_word(word, 'NNG', score=5)
print('Kiwi 사용자 사전 등록 완료')

# 역접 어미 패턴
CONTRAST_EC = {
    'ᆫ데', '은데', '는데', '지만',
    '더라도', '어도', '아도', '여도',
    '으나', '나',
}
CONTRAST_ADV = {
    '그러나', '그런데', '근데', '하지만', '다만', '반면',
    '그렇지만', '그래도', '그럼에도', '오히려', '도리어', '반대로',
}
ETM_NDE = {'ᆫ', '은', '는'}
MIN_SENT_LEN = 2

def split_sentences(text: str) -> list:
    if not text or not isinstance(text, str):
        return []
    try:
        kiwi_sents = kiwi.split_into_sents(text)
        sents = [s.text.strip() for s in kiwi_sents]
    except Exception:
        sents = [text]

    result = []
    for sent in sents:
        if len(sent) < MIN_SENT_LEN:
            continue
        try:
            tokens = kiwi.tokenize(sent)
        except Exception:
            result.append(sent)
            continue
        split_pos = []
        for idx, tok in enumerate(tokens):
            if tok.tag == 'EC' and tok.form in CONTRAST_EC:
                split_pos.append(tok.end)
            elif tok.tag == 'MAJ' and tok.form in CONTRAST_ADV:
                split_pos.append(tok.start)
            elif (tok.tag == 'ETM' and tok.form in ETM_NDE
                  and idx + 1 < len(tokens)
                  and tokens[idx+1].tag == 'NNB'
                  and tokens[idx+1].form == '데'):
                split_pos.append(tokens[idx+1].end)
        if not split_pos:
            result.append(sent)
            continue
        prev = 0
        for pos in sorted(set(split_pos)):
            chunk = sent[prev:pos].strip()
            chunk = re.sub(r'^[\s,\.·!?~\-]+', '', chunk).strip()
            if len(chunk) >= MIN_SENT_LEN:
                result.append(chunk)
            prev = pos
        tail = sent[prev:].strip()
        tail = re.sub(r'^[\s,\.·!?~\-]+', '', tail).strip()
        if len(tail) >= MIN_SENT_LEN:
            result.append(tail)
    return result if result else [text]

보호어: 104개
Kiwi 사용자 사전 등록 완료


In [ ]:
print('문장 분할 중...')
t0 = time.time()

tqdm.pandas()
df['sentences'] = df['리뷰내용_clean'].progress_apply(split_sentences)
df['n_sents']   = df['sentences'].apply(len)

elapsed = (time.time() - t0) / 60
print(f'분할 완료: {elapsed:.1f}분')
print(f'총 문장 수: {df["n_sents"].sum():,}')

# explode
META_COLS = [c for c in ['리뷰번호', 'topic_id', 'n_sents', '평점', 'goodsNo', '작성일'] if c in df.columns]

df_sents = (
    df[META_COLS + ['sentences']]
    .explode('sentences')
    .reset_index(drop=True)
    .rename(columns={'sentences': 'sentence'})
)
df_sents = df_sents[df_sents['sentence'].notna()].reset_index(drop=True)
df_sents = df_sents[df_sents['sentence'].str.len() >= MIN_SENT_LEN].reset_index(drop=True)

# n_sents 재계산 (필터링 후)
df_sents['n_sents'] = df_sents.groupby('리뷰번호')['sentence'].transform('count')
df_sents['sent_id'] = df_sents.groupby('리뷰번호').cumcount()

# 앞쪽 특수문자 정리
df_sents['sentence'] = df_sents['sentence'].str.replace(
    r'^[\s,\.·!?~\-]+', '', regex=True
).str.strip()
df_sents = df_sents[df_sents['sentence'].str.len() >= MIN_SENT_LEN].reset_index(drop=True)

output_path = DATA2_DIR + '4m_step7_1_sentences.parquet'
df_sents.to_parquet(output_path, index=False)
print(f'저장 완료: {output_path}')
print(f'  총 {len(df_sents):,}문장')

문장 분할 중...


  0%|          | 0/2752 [00:00<?, ?it/s]

분할 완료: 0.1분
총 문장 수: 7,512
저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_1_sentences.parquet
  총 7,512문장


## 5. Step 7-2 — 측면 매칭

In [ ]:
ASPECT_DICT = {
    'fit': {
        '오버핏', '오버사이즈', '루즈핏', '레귤러핏', '슬림핏', '와이드핏',
        '세미오버', '박시핏', '머슬핏', '크롭핏', '박시', '핏', '핏감',
        '벌룬핏', '부츠컷', '스탠다드핏', '세미와이드',
    },
    'size': {
        '사이즈', '치수', '크기', '길이', '품',
        '정사이즈', '큼', '작', '크', '맞',
        '어깨', '허리', '허벅지', '엉덩이',
        '짧', '넉넉', '타이트',
    },
    'fabric': {
        '재질', '원단', '소재', '촉감',
        '두께', '두께감', '두툼', '도톰', '두껍', '얇',
        '기모', '면', '폴리', '코듀로이', '니트', '데님',
        '신축', '스판', '탄성', '통기',
    },
    'color': {
        '색감', '색상', '컬러', '톤', '채도',
        '밝', '어둡', '진하', '연하', '선명',
        '사진', '실물', '화면', '보정',
    },
    'value': {
        '가성비', '가격', '퀄리티', '품질',
        '저렴', '비싸', '합리적', '가심비',
    },
    'detail': {
        '이쁘', '예쁘', '귀엽', '세련',
        '디자인', '프린팅', '자수', '패턴',
        '지퍼', '단추', '포켓', '주머니', '시보리',
    },
}

KEYWORD_TO_ASPECTS = {}
for cat, kws in ASPECT_DICT.items():
    for kw in kws:
        KEYWORD_TO_ASPECTS.setdefault(kw, []).append(cat)

ALLOWED_POS = {'NNG', 'NNP', 'VA', 'XR', 'SL', 'VV'}

def match_aspects(text: str) -> list:
    if not text or not isinstance(text, str):
        return []
    try:
        tokens = kiwi.tokenize(text)
    except Exception:
        return []
    matched = set()
    for tok in tokens:
        if not (tok.tag.startswith(('NN', 'VA', 'VV', 'XR')) or tok.tag == 'SL'):
            continue
        cats = KEYWORD_TO_ASPECTS.get(tok.form)
        if cats:
            for cat in cats:
                matched.add((cat, tok.form))
    return list(matched)

print('측면 매칭 중...')
t0 = time.time()

df_sents['matched_aspects'] = df_sents['sentence'].progress_apply(match_aspects)
df_sents['n_aspects'] = df_sents['matched_aspects'].apply(len)

elapsed = (time.time() - t0) / 60
n_matched = (df_sents['n_aspects'] >= 1).sum()
print(f'매칭 완료: {elapsed:.1f}분')
print(f'  매칭 성공: {n_matched:,}건 ({n_matched/len(df_sents)*100:.1f}%)')
print(f'  매칭 실패: {len(df_sents)-n_matched:,}건')

측면 매칭 중...


  0%|          | 0/7512 [00:00<?, ?it/s]

매칭 완료: 0.1분
  매칭 성공: 3,799건 (50.6%)
  매칭 실패: 3,713건


In [ ]:
# explode → (문장, 측면) 페어 생성
df_matched = df_sents[df_sents['n_aspects'] >= 1].copy()

front_cols = [c for c in ['리뷰번호', 'sent_id', 'sentence', 'n_sents', 'topic_id'] if c in df_matched.columns]
other_meta = [c for c in df_matched.columns
              if c not in front_cols + ['matched_aspects', 'n_aspects', 'sent_len']]

df_pairs = (
    df_matched[front_cols + other_meta + ['matched_aspects']]
    .explode('matched_aspects')
    .reset_index(drop=True)
)
df_pairs[['aspect_category', 'aspect_keyword']] = pd.DataFrame(
    df_pairs['matched_aspects'].tolist(), index=df_pairs.index
)
df_pairs = df_pairs.drop(columns=['matched_aspects'])

pairs_path = DATA2_DIR + '4m_step7_2_aspect_pairs.parquet'
df_pairs.to_parquet(pairs_path, index=False)

print(f'저장 완료: {pairs_path}')
print(f'  총 {len(df_pairs):,}페어')

저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_2_aspect_pairs.parquet
  총 5,801페어


## 6. Step 7-3b — 감성 분류 (KcELECTRA)

In [ ]:
# 라벨 매핑
KCELECTRA_POSITIVE = {'기쁨(행복한)', '고마운', '설레는(기대하는)', '사랑하는', '즐거운(신나는)'}
KCELECTRA_NEGATIVE = {'슬픔(우울한)', '힘듦(지침)', '짜증남', '걱정스러운(불안한)'}
KCELECTRA_NEUTRAL  = {'일상적인', '생각이 많은'}

MODEL_ID = 'nlp04/korean_sentiment_analysis_kcelectra'
print(f'모델 로드: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

id2label = model.config.id2label
POS_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_POSITIVE]
NEG_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_NEGATIVE]
NEU_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_NEUTRAL]

print(f'POS_IDX: {POS_IDX}')
print(f'NEG_IDX: {NEG_IDX}')
print(f'NEU_IDX: {NEU_IDX}')

모델 로드: nlp04/korean_sentiment_analysis_kcelectra


tokenizer_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/511M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

POS_IDX: [0, 1, 2, 3, 4]
NEG_IDX: [7, 8, 9, 10]
NEU_IDX: [5, 6]


In [ ]:
@torch.no_grad()
def predict_sentiment_probs(texts, batch_size=64, max_length=128):
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='추론'):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=max_length, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        probs_all = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        p_pos = probs_all[:, POS_IDX].sum(axis=1)
        p_neg = probs_all[:, NEG_IDX].sum(axis=1)
        p_neu = probs_all[:, NEU_IDX].sum(axis=1)
        all_probs.append(np.stack([p_pos, p_neg, p_neu], axis=1))
    return np.vstack(all_probs)

def assign_labels(probs_array, threshold_low):
    p_pos = probs_array[:, 0]
    p_neg = probs_array[:, 1]
    max_pn = np.maximum(p_pos, p_neg)
    labels = np.where(max_pn < threshold_low, 'neutral',
              np.where(p_pos > p_neg, 'positive', 'negative'))
    return labels

# 기존 threshold 로드
th_path = DATA2_DIR + 'step7_3b_thresholds.json'
if os.path.exists(th_path):
    with open(th_path, 'r') as f:
        th_info = json.load(f)
    THRESHOLD_LOW = th_info['threshold_low']
    print(f'기존 threshold 로드: {THRESHOLD_LOW:.3f}')
else:
    THRESHOLD_LOW = 0.35  # 기본값
    print(f'threshold 파일 없음. 기본값 사용: {THRESHOLD_LOW}')

threshold 파일 없음. 기본값 사용: 0.35


In [ ]:
# 트랙 A: 측면별 감성 분류
print('트랙 A: 측면별 감성 분류 중...')

CHECKPOINT_PATH = DATA2_DIR + '4m_step7_3b_track_a_checkpoint.parquet'
BATCH_SIZE = 256
SAVE_EVERY = 10000

df_unique = (
    df_pairs[['리뷰번호', 'sent_id', 'sentence']]
    .drop_duplicates(subset=['리뷰번호', 'sent_id'])
    .reset_index(drop=True)
)
print(f'고유 문장: {len(df_unique):,}건')

start_idx = 0
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_parquet(CHECKPOINT_PATH)
    start_idx = len(df_done)
    print(f'체크포인트: {start_idx:,}건 완료. 재개합니다.')
else:
    df_done = pd.DataFrame()

texts_all = df_unique['sentence'].tolist()
N = len(texts_all)
result_chunks = [df_done] if len(df_done) > 0 else []

for chunk_start in range(start_idx, N, SAVE_EVERY):
    chunk_end = min(chunk_start + SAVE_EVERY, N)
    chunk_texts = texts_all[chunk_start:chunk_end]

    print(f'\n청크 {chunk_start:,} ~ {chunk_end:,}')
    chunk_probs = predict_sentiment_probs(chunk_texts, batch_size=BATCH_SIZE)

    chunk_df = df_unique.iloc[chunk_start:chunk_end].copy()
    chunk_df['p_pos'] = chunk_probs[:, 0]
    chunk_df['p_neg'] = chunk_probs[:, 1]
    chunk_df['p_neu'] = chunk_probs[:, 2]
    chunk_df['sentiment'] = assign_labels(chunk_probs, THRESHOLD_LOW)

    result_chunks.append(chunk_df)
    df_done = pd.concat(result_chunks, ignore_index=True)
    df_done.to_parquet(CHECKPOINT_PATH, index=False)
    print(f'체크포인트 저장: {len(df_done):,}건')

# 페어와 join
df_unique_pred = df_done[['리뷰번호', 'sent_id', 'p_pos', 'p_neg', 'p_neu', 'sentiment']]
df_aspect_sentiment = df_pairs.merge(df_unique_pred, on=['리뷰번호', 'sent_id'], how='left')

absa_path = DATA2_DIR + '4m_step7_3b_aspect_sentiment.parquet'
df_aspect_sentiment.to_parquet(absa_path, index=False)
print(f'\n저장 완료: {absa_path}')
print(f'  총 {len(df_aspect_sentiment):,}페어')
print(f'\n[측면별 감성 분포]')
print(df_aspect_sentiment['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

트랙 A: 측면별 감성 분류 중...
고유 문장: 3,799건
체크포인트: 3,101건 완료. 재개합니다.

청크 3,101 ~ 3,799


추론:   0%|          | 0/3 [00:00<?, ?it/s]

체크포인트 저장: 3,799건

저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_3b_aspect_sentiment.parquet
  총 6,694페어

[측면별 감성 분포]
sentiment
positive    68.7%
neutral     16.0%
negative    15.3%
Name: proportion, dtype: object


In [ ]:
# 트랙 B: 리뷰 단위 감성 분류
print('트랙 B: 리뷰 단위 감성 분류 중...')

TRACK_B_CHECKPOINT = DATA2_DIR + '4m_step7_3b_track_b_checkpoint.parquet'

start_idx_b = 0
if os.path.exists(TRACK_B_CHECKPOINT):
    df_done_b = pd.read_parquet(TRACK_B_CHECKPOINT)
    start_idx_b = len(df_done_b)
    print(f'체크포인트: {start_idx_b:,}건 완료. 재개합니다.')
else:
    df_done_b = pd.DataFrame()

texts_b = df['리뷰내용_clean'].tolist()
N_B = len(texts_b)
print(f'전체: {N_B:,} | 시작: {start_idx_b:,}')

result_chunks_b = [df_done_b] if len(df_done_b) > 0 else []

for chunk_start in range(start_idx_b, N_B, SAVE_EVERY):
    chunk_end = min(chunk_start + SAVE_EVERY, N_B)
    chunk_texts = texts_b[chunk_start:chunk_end]

    print(f'\n청크 {chunk_start:,} ~ {chunk_end:,}')
    chunk_probs = predict_sentiment_probs(chunk_texts, batch_size=BATCH_SIZE)

    chunk_df = df.iloc[chunk_start:chunk_end].copy()
    chunk_df['p_pos'] = chunk_probs[:, 0]
    chunk_df['p_neg'] = chunk_probs[:, 1]
    chunk_df['p_neu'] = chunk_probs[:, 2]
    chunk_df['sentiment'] = assign_labels(chunk_probs, THRESHOLD_LOW)

    result_chunks_b.append(chunk_df)
    df_done_b = pd.concat(result_chunks_b, ignore_index=True)
    df_done_b.to_parquet(TRACK_B_CHECKPOINT, index=False)
    print(f'체크포인트 저장: {len(df_done_b):,}건')

review_sent_path = DATA2_DIR + '4m_step7_3b_review_sentiment.parquet'
df_done_b.to_parquet(review_sent_path, index=False)
print(f'\n저장 완료: {review_sent_path}')
print(f'  총 {len(df_done_b):,}건')
print(f'\n[리뷰 단위 감성 분포]')
print(df_done_b['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

트랙 B: 리뷰 단위 감성 분류 중...
체크포인트: 2,234건 완료. 재개합니다.
전체: 2,752 | 시작: 2,234

청크 2,234 ~ 2,752


추론:   0%|          | 0/3 [00:00<?, ?it/s]

체크포인트 저장: 2,752건

저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_3b_review_sentiment.parquet
  총 2,752건

[리뷰 단위 감성 분포]
sentiment
positive    86.0%
negative    10.0%
neutral      4.0%
Name: proportion, dtype: object


## 7. 결과 검증

In [ ]:
print('='*55)
print('4월 데이터 검증 결과 요약')
print('='*55)
print(f'전체 리뷰: {len(df):,}건')
print(f'토픽 배정: {(cluster_labels != -1).sum():,}건 ({(cluster_labels != -1).mean()*100:.1f}%)')
print(f'아웃라이어: {(cluster_labels == -1).sum():,}건 ({(cluster_labels == -1).mean()*100:.1f}%)')

print(f'\n[토픽 배정 감성 분포]')
topic_df = df_done_b[df_done_b['topic_id'] != -1]
print(topic_df['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

print(f'\n[아웃라이어 감성 분포]')
outlier_df = df_done_b[df_done_b['topic_id'] == -1]
print(outlier_df['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

if '평점' in df_done_b.columns:
    print(f'\n[평점 vs 감성 교차표 (%)]')
    cross = pd.crosstab(
        df_done_b['평점'].fillna(-1).astype(int),
        df_done_b['sentiment'],
        normalize='index'
    ) * 100
    print(cross.round(1))

4월 데이터 검증 결과 요약
전체 리뷰: 2,752건
토픽 배정: 1,732건 (62.9%)
아웃라이어: 1,020건 (37.1%)

[토픽 배정 감성 분포]
sentiment
positive    88.0%
negative     9.0%
neutral      3.0%
Name: proportion, dtype: object

[아웃라이어 감성 분포]
sentiment
positive    82.6%
negative    11.7%
neutral      5.6%
Name: proportion, dtype: object

[평점 vs 감성 교차표 (%)]
sentiment  negative  neutral  positive
평점                                    
1              75.0      5.0      20.0
2              86.7      0.0      13.3
3              43.0      5.1      51.9
4              29.0      9.9      61.1
5               5.5      3.2      91.3


## 8. 대분류 추가

In [ ]:
# topic_summary 로드
df_topic_summary = pd.read_csv(DATA_DIR + 'topic_summary.csv')
df_topic_summary = df_topic_summary[['topic_id', 'topic_category', 'topic_name']]

# aspect_sentiment에 붙이기
df_asp_final = df_aspect_sentiment.merge(df_topic_summary, on='topic_id', how='left')
df_asp_final['topic_category'] = df_asp_final['topic_category'].fillna('아웃라이어')
df_asp_final['topic_name'] = df_asp_final['topic_name'].fillna('아웃라이어')

asp_csv_path = DATA2_DIR + '4m_step7_3b_aspect_sentiment_with_topic.csv'
df_asp_final.to_csv(asp_csv_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {asp_csv_path} ({len(df_asp_final):,}건)')
display(df_asp_final.head())

# review_sentiment에 붙이기
df_rev_final = df_done_b.merge(df_topic_summary, on='topic_id', how='left')
df_rev_final['topic_category'] = df_rev_final['topic_category'].fillna('아웃라이어')
df_rev_final['topic_name'] = df_rev_final['topic_name'].fillna('아웃라이어')

rev_csv_path = DATA2_DIR + '4m_step7_3b_review_sentiment_with_topic.csv'
df_rev_final.to_csv(rev_csv_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {rev_csv_path} ({len(df_rev_final):,}건)')
display(df_rev_final.head())

저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_3b_aspect_sentiment_with_topic.csv (6,694건)


,리뷰번호,sent_id,sentence,n_sents,topic_id,평점,goodsNo,작성일,aspect_category,aspect_keyword,p_pos,p_neg,p_neu,sentiment,topic_category,topic_name
0,83514644,0,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,1,-1,5,3791889,2026-04-07 21:01:22,fabric,면,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
1,83514644,0,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,1,-1,5,3791889,2026-04-07 21:01:22,color,선명,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
2,83514644,0,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,1,-1,5,3791889,2026-04-07 21:01:22,color,색감,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
3,83514617,0,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,1,-1,5,3791890,2026-04-07 21:00:20,fabric,면,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
4,83514617,0,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,1,-1,5,3791890,2026-04-07 21:00:20,color,선명,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어


저장 완료: /content/drive/MyDrive/review_analysis/4m_data/4m_step7_3b_review_sentiment_with_topic.csv (2,752건)


,리뷰번호,리뷰내용_clean,goodsNo,리뷰타입,평점,작성일,사이즈,화면대비색감,퀄리티,구김,...,체험단,topic_id,sentences,n_sents,p_pos,p_neg,p_neu,sentiment,topic_category,topic_name
0,83514644,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,3791889,general,5,2026-04-07 21:01:22,조금 큼,화면과 비슷,보통,약간 있음,...,False,-1,[면이 탄탄한 느낌이고 색감도 선명하고 좋습니다],1,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
1,83514617,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,3791890,general,5,2026-04-07 21:00:20,조금 큼,화면과 비슷,보통,약간 있음,...,False,-1,[면이 탄탄한 느낌이고 색감도 선명하고 좋습니다],1,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
2,83514598,면이 탄탄한 느낌이고 색감도 선명하고 좋습니다,3791891,general,5,2026-04-07 20:59:46,조금 큼,화면과 비슷,보통,약간 있음,...,False,-1,[면이 탄탄한 느낌이고 색감도 선명하고 좋습니다],1,0.970675,0.007354,0.021971,positive,아웃라이어,아웃라이어
3,83784232,제 교복템이 되어버렸습니다! 적당한 길이에 적당한 크기에여 근데 안에 안감?이 꽤나...,5923260,general,5,2026-04-18 21:14:57,정사이즈,화면과 비슷,보통,None,...,False,29,"[제 교복템이 되어버렸습니다!, 적당한 길이에 적당한 크기에여, 근데 안에 안감?,...",5,0.803575,0.110823,0.085602,positive,시즌/날씨,여름 두께감
4,83408119,디자인이 흔하지 않아서 이뻐용 근데 색이 은근 초록빛이 많이 돌아요 모델분이 입은 ...,3455882,general,2,2026-04-11 19:15:26,많이 큼,어두움,보통,None,...,False,46,"[디자인이 흔하지 않아서 이뻐용, 근데 색이 은근 초록빛이 많이 돌아요, 모델분이 ...",5,0.562228,0.254526,0.183246,positive,사이즈/핏,L/M/S 사이즈
